# G2P-RAG Evaluation
Retrieval and generation quality on 20 hand-written QA pairs.

In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path("..") / "src"))

from g2p_rag.fetch import fetch_all_genes, GENE_LIST
from g2p_rag.chunk import chunk_all, ProteinChunker
from g2p_rag.embed import get_embedder
from g2p_rag.retrieve import build_index, HybridRetriever, SearchResult
from g2p_rag.generate import build_chain

DATA_DIR = Path("../data")
CHROMA_DIR = DATA_DIR / "chroma"
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

## 20 QA Pairs

In [ ]:
QA_PAIRS = [
    # --- Domain questions (5) ---
    {
        "question": "What domains does BRCA1 contain?",
        "expected_genes": ["BRCA1"],
        "expected_chunk_types": ["domain", "protein_summary"],
        "answer_keywords": ["RING", "BRCT", "coiled-coil", "domain"],
    },
    {
        "question": "Which EGFR domains are mutated in lung cancer variants?",
        "expected_genes": ["EGFR"],
        "expected_chunk_types": ["domain", "variant_cluster"],
        "answer_keywords": ["kinase", "EGFR", "L858R", "exon"],
    },
    {
        "question": "What is the kinase domain of BRAF?",
        "expected_genes": ["BRAF"],
        "expected_chunk_types": ["domain"],
        "answer_keywords": ["kinase", "BRAF", "serine", "threonine"],
    },
    {
        "question": "Where is the PH domain in PIK3CA?",
        "expected_genes": ["PIK3CA"],
        "expected_chunk_types": ["domain"],
        "answer_keywords": ["PH", "PIK3CA", "pleckstrin", "residue"],
    },
    {
        "question": "Describe the APC armadillo repeat domain.",
        "expected_genes": ["APC"],
        "expected_chunk_types": ["domain"],
        "answer_keywords": ["armadillo", "APC", "repeat", "beta-catenin"],
    },
    # --- Variant cluster questions (5) ---
    {
        "question": "What pathogenic variants cluster around residue 61 in BRCA1?",
        "expected_genes": ["BRCA1"],
        "expected_chunk_types": ["variant_cluster"],
        "answer_keywords": ["BRCA1", "pathogenic", "missense", "residue"],
    },
    {
        "question": "Which KRAS mutations are at codon 12?",
        "expected_genes": ["KRAS"],
        "expected_chunk_types": ["variant_cluster"],
        "answer_keywords": ["G12", "KRAS", "codon", "Gly"],
    },
    {
        "question": "What TP53 variants occur in the DNA-binding domain?",
        "expected_genes": ["TP53"],
        "expected_chunk_types": ["variant_cluster", "domain"],
        "answer_keywords": ["TP53", "DNA-binding", "R175H", "hotspot"],
    },
    {
        "question": "List pathogenic PTEN variants in the phosphatase domain.",
        "expected_genes": ["PTEN"],
        "expected_chunk_types": ["variant_cluster", "domain"],
        "answer_keywords": ["PTEN", "phosphatase", "pathogenic", "catalytic"],
    },
    {
        "question": "What are the common CFTR pathogenic variants?",
        "expected_genes": ["CFTR"],
        "expected_chunk_types": ["variant_cluster", "protein_summary"],
        "answer_keywords": ["CFTR", "F508del", "cystic fibrosis", "pathogenic"],
    },
    # --- PTM / PPI questions (4) ---
    {
        "question": "What phosphorylation sites does TP53 have?",
        "expected_genes": ["TP53"],
        "expected_chunk_types": ["domain", "protein_summary"],
        "answer_keywords": ["phosphorylation", "Ser15", "TP53", "PTM"],
    },
    {
        "question": "Which proteins interact with BRCA2?",
        "expected_genes": ["BRCA2"],
        "expected_chunk_types": ["domain", "protein_summary"],
        "answer_keywords": ["BRCA2", "RAD51", "PALB2", "interaction"],
    },
    {
        "question": "What ubiquitination sites are in MUC1?",
        "expected_genes": ["MUC1"],
        "expected_chunk_types": ["domain", "protein_summary"],
        "answer_keywords": ["ubiquitination", "MUC1", "lysine", "PTM"],
    },
    {
        "question": "What are the key PTMs in EGFR kinase domain?",
        "expected_genes": ["EGFR"],
        "expected_chunk_types": ["domain"],
        "answer_keywords": ["EGFR", "phosphorylation", "Tyr", "autophosphorylation"],
    },
    # --- MaveDB / functional questions (3) ---
    {
        "question": "What MaveDB scores are available for BRCA1 variants?",
        "expected_genes": ["BRCA1"],
        "expected_chunk_types": ["domain", "protein_summary"],
        "answer_keywords": ["MaveDB", "BRCA1", "score", "functional"],
    },
    {
        "question": "Which PTEN variants have loss-of-function scores?",
        "expected_genes": ["PTEN"],
        "expected_chunk_types": ["domain", "variant_cluster"],
        "answer_keywords": ["PTEN", "loss-of-function", "score", "MaveDB"],
    },
    {
        "question": "What is the functional impact of common KRAS G12 mutations?",
        "expected_genes": ["KRAS"],
        "expected_chunk_types": ["variant_cluster", "domain"],
        "answer_keywords": ["KRAS", "G12", "GTPase", "activating"],
    },
    # --- Multi-gene / comparative questions (3) ---
    {
        "question": "Compare the domain structures of MLH1 and MSH2.",
        "expected_genes": ["MLH1", "MSH2"],
        "expected_chunk_types": ["domain", "protein_summary"],
        "answer_keywords": ["MLH1", "MSH2", "mismatch repair", "domain"],
    },
    {
        "question": "What genes have druggable pockets?",
        "expected_genes": ["KRAS", "EGFR", "BRAF"],
        "expected_chunk_types": ["domain", "protein_summary"],
        "answer_keywords": ["druggable", "pocket", "binding", "inhibitor"],
    },
    {
        "question": "Which cancer genes have the most pathogenic variants in ClinVar?",
        "expected_genes": ["BRCA1", "BRCA2", "TP53"],
        "expected_chunk_types": ["protein_summary"],
        "answer_keywords": ["ClinVar", "pathogenic", "BRCA1", "TP53"],
    },
]

In [ ]:
# Load retriever — assumes ingest has been run
from g2p_rag.retrieve import load_retriever, load_embedder, CollectionEmptyError

try:
    embedder = load_embedder(EMBEDDING_MODEL)
    retriever = load_retriever(CHROMA_DIR, embedder)
    print(f"Loaded retriever with {retriever._vs.count()} chunks")
except CollectionEmptyError:
    print("No data — run: g2p-rag ingest")
    retriever = None

## Retrieval Evaluation (Recall@5, MRR)

In [ ]:
def recall_at_k(results: list, expected_genes: list, k: int = 5) -> float:
    """Fraction of expected genes found in top-k results."""
    top_genes = {r.chunk.gene for r in results[:k]}
    return len(set(expected_genes) & top_genes) / len(expected_genes)

def reciprocal_rank(results: list, expected_genes: list) -> float:
    """1/rank of first result from an expected gene."""
    for i, r in enumerate(results, 1):
        if r.chunk.gene in expected_genes:
            return 1.0 / i
    return 0.0

retrieval_results = []
if retriever:
    for qa in QA_PAIRS:
        results = retriever.search(qa["question"], k=5)
        r5 = recall_at_k(results, qa["expected_genes"])
        rr = reciprocal_rank(results, qa["expected_genes"])
        retrieval_results.append({"question": qa["question"][:60], "recall@5": r5, "mrr": rr})

import statistics
if retrieval_results:
    avg_r5 = statistics.mean(r["recall@5"] for r in retrieval_results)
    avg_mrr = statistics.mean(r["mrr"] for r in retrieval_results)
    print(f"Mean Recall@5: {avg_r5:.3f}")
    print(f"Mean MRR:      {avg_mrr:.3f}")

## Generation Evaluation (Citation Accuracy)

In [ ]:
import re

chain = build_chain() if retriever else None

def citation_accuracy(answer: str, expected_genes: list) -> float:
    """Fraction of expected genes cited in the answer."""
    cited = set(re.findall(r"\[([A-Z0-9]+):[^\]]+\]", answer))
    return len(set(expected_genes) & cited) / len(expected_genes)

gen_results = []
if chain and retriever:
    for qa in QA_PAIRS[:5]:
        results = retriever.search(qa["question"], k=5)
        gen = chain.answer(qa["question"], results)
        acc = citation_accuracy(gen.answer, qa["expected_genes"])
        gen_results.append({"question": qa["question"][:60], "citation_accuracy": acc})
    
    avg_acc = statistics.mean(r["citation_accuracy"] for r in gen_results)
    print(f"Mean Citation Accuracy (5 QA): {avg_acc:.3f}")

## Results Summary

In [ ]:
print("\n=== Retrieval Results ===")
print(f"{'Question':<62} {'R@5':>5} {'MRR':>5}")
print("-" * 75)
for r in retrieval_results:
    print(f"{r['question']:<62} {r['recall@5']:>5.2f} {r['mrr']:>5.2f}")

print("\n=== Generation Results ===")
print(f"{'Question':<62} {'Cite Acc':>8}")
print("-" * 72)
for r in gen_results:
    print(f"{r['question']:<62} {r['citation_accuracy']:>8.2f}")